# Training : Textual Inversion

-----
- Conda env : [cv_playgrounds](../README.md#setup-a-conda-environment)
-----

- Ref: https://huggingface.co/docs/diffusers/v0.34.0/en/training/text_inversion

In [1]:
#@title
import torch

if torch.backends.mps.is_available():
    t_device = torch.device("mps")
    s_device = "mps"
    print(f"Current memory allocated on MPS: {torch.mps.current_allocated_memory()} bytes")
    print(f"Driver memory allocated on MPS: {torch.mps.driver_allocated_memory()} bytes")
elif torch.cuda.is_available():
    t_device = torch.device("cuda")
    s_device = "cuda"
else:
    t_device = torch.device("cpu")
    s_device = "cpu"
print(s_device)

Current memory allocated on MPS: 0 bytes
Driver memory allocated on MPS: 393216 bytes
mps


### 6. Test

In [ ]:
from diffusers import StableDiffusionPipeline

pipeline_origin = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5").to(t_device)

pipeline_object = StableDiffusionPipeline.from_pipe(pipeline_origin)
pipeline_object.load_textual_inversion("./temp/textual_inversion_cat")

pipeline_style = StableDiffusionPipeline.from_pipe(pipeline_origin)
pipeline_style.load_textual_inversion("./temp/textual_inversion_cat_style")


In [ ]:
sample_prompts = ["A monalisa of toy",
                  "A monalisa with the style of toy",
                  "A clock of toy",
                  "A toy clock",
                  "A toy box",
                  "A box of toy",
                  "A toy flower",
                  "A flower of toy",
                  "A monalisa of <cat-toy>",
                  "A monalisa with the style of <cat-toy>",
                  "A clock of <cat-toy>",
                  "A <cat-toy> clock",
                  "A <cat-toy> box",
                  "A box of <cat-toy>",
                  "A <cat-toy> flower",
                  "A flower of <cat-toy>",
                  "A monalisa of <cat-style-toy>",
                  "A monalisa with the style of <cat-style-toy>",
                  "A clock of <cat-style-toy>",
                  "A <cat-style-toy> clock",
                  "A <cat-style-toy> box",
                  "A box of <cat-style-toy>",
                  "A <cat-style-toy> flower",
                  "A flower of <cat-style-toy>"
                ]
sample_nums = 4

#### 6.1. Sampling with the original pipeline

In [ ]:


original_samples = {}
object_samples = {}
style_samples = {}
for prompt in sample_prompts:
    print(f"\n {prompt}")
    original_samples[prompt] = pipeline_origin(prompt, num_inference_steps=50, num_images_per_prompt=sample_nums).images
    object_samples[prompt] = pipeline_object(prompt, num_inference_steps=50, num_images_per_prompt=sample_nums).images
    style_samples[prompt] = pipeline_style(prompt, num_inference_steps=50, num_images_per_prompt=sample_nums).images



In [ ]:
print(len(original_samples))
print(len(style_samples))
print(len(object_samples))

In [ ]:
import matplotlib.pyplot as plt

img_keys = list(original_samples.keys())
im_k = img_keys[0]

for im_k in original_samples.keys():
    # fig, axs = plt.subplots(3, 4, figsize=(16, 4))
    fig, axs = plt.subplots(3, 4)
    for i, image in enumerate(original_samples[im_k]):
        axs[0, i].imshow(image)
        axs[0, i].set_axis_off()
    axs[0, 0].set_title("origin")
    for i, image in enumerate(object_samples[im_k]):
        axs[1, i].imshow(image)
        axs[1, i].set_axis_off()
    axs[1, 0].set_title("object")
    for i, image in enumerate(style_samples[im_k]):
        axs[2, i].imshow(image)
        axs[2, i].set_axis_off()
    axs[2, 0].set_title("style")
    fig.suptitle(f"{im_k}")
    fig.show()